In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time  # <--- Ajouté pour mesurer le temps d'exécution

# =============================================================================
# DEBUT DU CHRONOMÈTRE GLOBAL
# =============================================================================
start_global_time = time.time()

# =============================================================================
# PARTIE 1 : CHARGEMENT DES DONNÉES & PRÉPARATION
# =============================================================================

path = r"C:\Users\pc\projet\return_nasdaq.m"
path_cov = r"C:\Users\pc\projet\covariance_nasdaq.m" 

try:
    data_returns = pd.read_csv(path, sep=r"\s+", header=None, skiprows=1, on_bad_lines='skip', engine='python')
    benefice = data_returns.to_numpy()
    print("--- Loading Returns Successful ---")
    print(f"Matrix Dimensions: {benefice.shape}")
except Exception as e:
    print(f"Error loading returns: {e}")
    benefice = np.random.randn(2196, 1)

try:
    data_cov = pd.read_csv(path_cov, sep=r"\s+", header=None, skiprows=1, on_bad_lines='skip', engine='python')
    covariance = data_cov.to_numpy()
    print("--- Loading Covariance Successful ---")
    print(f"Matrix Dimensions: {covariance.shape}")
except Exception as e:
    print(f"Error loading covariance: {e}")
    covariance = np.diag(np.random.uniform(0.01, 0.05, 2196))

# Configuration des paramètres de l'univers d'actifs
nb_asset = 2196
cardinal = 300

return_vect = benefice.flatten()
covar_matrix = covariance
variability = np.sqrt(np.diag(covar_matrix))

# Définition de la cible (Actifs avec rendement supérieur à la moyenne)
seuil = np.mean(return_vect)
target = (return_vect >= seuil).astype(int)

# Préparation des features pour le MLP
X = np.column_stack((return_vect, variability))
y = target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =============================================================================
# PARTIE 2 : CONFIGURATION ET ENTRAÎNEMENT DU MLP
# =============================================================================

model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

print("\n--- Entraînement du MLP ---")
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)

loss, accuracy, precision, recall = model.evaluate(X_test, y_test, verbose=0)
print(f"Précision du MLP sur le test set : {accuracy:.2f}") 

# Prédiction sur l'ensemble des actifs
X_all_scaled = scaler.transform(X)
predictions_finales = (model.predict(X_all_scaled, verbose=0) > 0.5).astype(int).flatten()

# Sélection des indices retenus par le MLP
index_vector = [i for i in range(len(predictions_finales)) if predictions_finales[i] == 1]

# Filtrage par rendement pour ne garder que le top 'cardinal' (300)
return_index_vector = return_vect[index_vector]
indices_tries = sorted(range(len(return_index_vector)), key=lambda i: return_index_vector[i], reverse=True)
top_indices = indices_tries[:cardinal]
index_vector_vector = [index_vector[i] for i in top_indices]

# Création du masque binaire final
vecteur_portefeuille = np.zeros(nb_asset)
for i in index_vector_vector:
    vecteur_portefeuille[i] = 1

print("\n--- Résultats du filtrage MLP ---")
print(f"Nombre d'actifs sélectionnés : {len(index_vector_vector)}")

# =============================================================================
# PARTIE 3 : CODE NSGA-II OPTIMISÉ POUR LA DIVERSITÉ
# =============================================================================
print("\n--- Lancement de l'optimisation NSGA-II ---")

return_vect_filtered = return_vect.copy()
covar_matrix_filtered = covar_matrix.copy()

# Isolation stricte des 300 actifs choisis par le MLP
for i in range(nb_asset):
    if vecteur_portefeuille[i] == 0:
        return_vect_filtered[i] = 0
        covar_matrix_filtered[i, :] = 0
        covar_matrix_filtered[:, i] = 0

# Paramètres de contrôle réglés pour maximiser la diversité
nb_sol = 150                  
number_iteration_max = 500     
prob_crossover = 0.95         
prob_mutation = 2 / cardinal  

# Initialisation de la population initiale (Somme des poids = 1)
population = np.zeros((nb_sol, nb_asset))
for i in range(nb_sol):
    random_weights = np.random.uniform(0.1, 1.0, size=len(index_vector_vector))
    population[i, index_vector_vector] = random_weights / np.sum(random_weights)

def evaluate_population(pop):
    returns = np.dot(pop, return_vect_filtered)
    risks = np.zeros(len(pop))
    for idx in range(len(pop)):
        w = pop[idx, :].reshape(1, -1)
        risks[idx] = w @ covar_matrix_filtered @ w.T
    return returns, risks

def fast_non_dominated_sort(returns, risks):
    pop_size = len(returns)
    domination_features = [[] for _ in range(pop_size)]
    domination_counters = np.zeros(pop_size, dtype=int)
    fronts = [[]]
    
    for p in range(pop_size):
        for q in range(pop_size):
            if (returns[p] >= returns[q] and risks[p] <= risks[q]) and (returns[p] > returns[q] or risks[p] < risks[q]):
                domination_features[p].append(q)
            elif (returns[q] >= returns[p] and risks[q] <= risks[p]) and (returns[q] > returns[p] or risks[q] < risks[p]):
                domination_counters[p] += 1
                
        if domination_counters[p] == 0:
            fronts[0].append(p)
            
    i = 0
    while len(fronts[i]) > 0:
        next_front = []
        for p in fronts[i]:
            for q in domination_features[p]:
                domination_counters[q] -= 1
                if domination_counters[q] == 0:
                    next_front.append(q)
        i += 1
        fronts.append(next_front)
    return fronts[:-1]

def calculate_crowding_distance(returns, risks, front):
    distance = np.zeros(len(front))
    if len(front) <= 2:
        distance[:] = np.inf
        return distance
        
    obj_returns = returns[front]
    sort_idx_ret = np.argsort(obj_returns)
    distance[sort_idx_ret[0]] = np.inf
    distance[sort_idx_ret[-1]] = np.inf
    ret_range = np.max(obj_returns) - np.min(obj_returns)
    if ret_range > 0:
        for i in range(1, len(front) - 1):
            distance[sort_idx_ret[i]] += (obj_returns[sort_idx_ret[i+1]] - obj_returns[sort_idx_ret[i-1]]) / ret_range
            
    obj_risks = risks[front]
    sort_idx_risk = np.argsort(obj_risks)
    distance[sort_idx_risk[0]] = np.inf
    distance[sort_idx_risk[-1]] = np.inf
    risk_range = np.max(obj_risks) - np.min(obj_risks)
    if risk_range > 0:
        for i in range(1, len(front) - 1):
            distance[sort_idx_risk[i]] += (obj_risks[sort_idx_risk[i+1]] - obj_risks[sort_idx_risk[i-1]]) / risk_range
            
    return distance

# Boucle principale de l'algorithme génétique (500 Générations)
for generation in range(1, number_iteration_max + 1):
    
    returns, risks = evaluate_population(population)
    offspring = np.zeros_like(population)
    
    indices = np.arange(nb_sol)
    np.random.shuffle(indices)
    
    for k in range(0, nb_sol, 2):
        parent1 = population[indices[k]]
        parent2 = population[indices[k+1]]
        child1 = parent1.copy()
        child2 = parent2.copy()
        
        if np.random.rand() < prob_crossover:
            alpha = np.random.uniform(-0.1, 1.1, size=nb_asset)
            child1 = alpha * parent1 + (1 - alpha) * parent2
            child2 = alpha * parent2 + (1 - alpha) * parent1
            
        for asset_idx in index_vector_vector:
            if np.random.rand() < prob_mutation:
                child1[asset_idx] += np.random.normal(0, 0.08)
            if np.random.rand() < prob_mutation:
                child2[asset_idx] += np.random.normal(0, 0.08)
                
        child1[child1 < 0] = 0
        child2[child2 < 0] = 0
        child1[vecteur_portefeuille == 0] = 0
        child2[vecteur_portefeuille == 0] = 0
        
        if np.sum(child1) > 0: child1 /= np.sum(child1)
        if np.sum(child2) > 0: child2 /= np.sum(child2)
        
        offspring[k] = child1
        offspring[k+1] = child2

    combined_pop = np.concatenate((population, offspring), axis=0)
    comb_returns, comb_risks = evaluate_population(combined_pop)
    
    fronts = fast_non_dominated_sort(comb_returns, comb_risks)
    first_front_indices = np.array(fronts[0])
    
    print(f"Iteration  {generation}")
    print(f"Non dominated selected solutions in iteration {generation} are: {first_front_indices}")
    print(f"Expected return of these solutions is: {comb_returns[first_front_indices]}")
    print(f"Risk of these solutions is: {comb_risks[first_front_indices]}\n")

    new_population = []
    front_idx = 0
    while len(new_population) + len(fronts[front_idx]) <= nb_sol:
        new_population.extend(fronts[front_idx])
        front_idx += 1
        if front_idx >= len(fronts): break
        
    if len(new_population) < nb_sol and front_idx < len(fronts):
        last_front = fronts[front_idx]
        distances = calculate_crowding_distance(comb_returns, comb_risks, last_front)
        sorted_last_front = [last_front[i] for i in np.argsort(distances)[::-1]]
        needed = nb_sol - len(new_population)
        new_population.extend(sorted_last_front[:needed])
        
    population = combined_pop[new_population]

# =============================================================================
# FIN DU CHRONOMÈTRE GLOBAL ET AFFICHAGE DES RÉSULTATS
# =============================================================================
end_global_time = time.time()
execution_time_seconds = end_global_time - start_global_time
execution_time_minutes = execution_time_seconds / 60

print("="*60)
print("--- OPTIMISATION TERMINÉE AVEC SUCCÈS ---")
print(f"Temps d'exécution global : {execution_time_seconds:.2f} secondes ({execution_time_minutes:.2f} minutes)")
print("="*60)

--- Loading Returns Successful ---
Matrix Dimensions: (2196, 1)
--- Loading Covariance Successful ---
Matrix Dimensions: (2196, 2196)

--- Entraînement du MLP ---
Précision du MLP sur le test set : 0.99

--- Résultats du filtrage MLP ---
Nombre d'actifs sélectionnés : 300

--- Lancement de l'optimisation NSGA-II ---
Iteration  1
Non dominated selected solutions in iteration 1 are: [ 44  47  86 150 196 205 206 207 219 237 282 293]
Expected return of these solutions is: [0.01109868 0.0107885  0.01109969 0.01114356 0.01092467 0.01106141
 0.01136991 0.01124692 0.01155663 0.01105688 0.01110037 0.01117895]
Risk of these solutions is: [0.00062697 0.00056748 0.00063729 0.00066527 0.00058957 0.00060172
 0.00072525 0.00071618 0.00075921 0.00059122 0.00066249 0.00070563]

Iteration  2
Non dominated selected solutions in iteration 2 are: [  1   8 162 177 214 219 255 275]
Expected return of these solutions is: [0.0107885  0.01155663 0.01130021 0.01066371 0.01143427 0.01193526
 0.0108955  0.01070878